# Algorithmic Trading and Quantitative Strategies
## Part 4: Introduction to Backtrader 
**Dr. Ayhan Yuksel, CFA, FDP, FRM, PRM**

Bogazici University, EC581

## Table of Contents

1. Vectorized Backtesting
2. Introduction to Backtrader
3. Golden Cross Strategy in Backtrader
4. Understanding the Output
5. Exercises

## 1. Vectorized Backtesting

Before using a dedicated backtesting framework, let us understand backtesting by implementing a simple strategy using only pandas. This is called *vectorized backtesting* because we use vectorized array operations rather than looping bar-by-bar.

### 1.1 The Golden Cross Strategy

The **Golden Cross** strategy is one of the most well-known technical trading signals:
- **Buy** when the short-term moving average (e.g., 50-day) crosses above the long-term moving average (e.g., 200-day)
- **Sell** when the short-term crosses below the long-term (*Death Cross*)

This is a **trend-following** strategy based on the idea that sustained momentum will continue.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

# Download SPY data
data = yf.download('SPY', start='2010-01-01', end='2024-12-31')
data.columns = data.columns.droplevel('Ticker')
close = data['Close']
print(f"Downloaded {len(close)} trading days for SPY")
close.tail()

### 1.2 Implementing the Strategy with Pandas

In [ ]:
# Parameters
short_window = 50
long_window = 200

# Calculate moving averages
df = pd.DataFrame({'Close': close})
df['SMA_50'] = df['Close'].rolling(short_window).mean()
df['SMA_200'] = df['Close'].rolling(long_window).mean()

# Generate signals: 1 = long, 0 = flat
df['Signal'] = 0
df.loc[df['SMA_50'] > df['SMA_200'], 'Signal'] = 1

# Position: shifted signal to avoid look-ahead bias
df['Position'] = df['Signal'].shift(1)

# Returns
df['Market_Return'] = df['Close'].pct_change()
df['Strategy_Return'] = df['Position'] * df['Market_Return']

# Drop NaN rows
df = df.dropna()

# Cumulative returns
df['Market_Cumulative'] = (1 + df['Market_Return']).cumprod()
df['Strategy_Cumulative'] = (1 + df['Strategy_Return']).cumprod()

print(f"Market Total Return:   {df['Market_Cumulative'].iloc[-1] - 1:.2%}")
print(f"Strategy Total Return: {df['Strategy_Cumulative'].iloc[-1] - 1:.2%}")

In [ ]:
# Plot
fig, axes = plt.subplots(3, 1, figsize=(14, 12), gridspec_kw={'height_ratios': [3, 1, 3]})

# Price + Moving Averages
axes[0].plot(df['Close'], label='SPY Close', alpha=0.7)
axes[0].plot(df['SMA_50'], label=f'SMA({short_window})', linewidth=1.5)
axes[0].plot(df['SMA_200'], label=f'SMA({long_window})', linewidth=1.5)
buy_signals = df[df['Signal'].diff() == 1]
sell_signals = df[df['Signal'].diff() == -1]
axes[0].scatter(buy_signals.index, buy_signals['Close'], marker='^', color='green', s=80, label='Buy', zorder=5)
axes[0].scatter(sell_signals.index, sell_signals['Close'], marker='v', color='red', s=80, label='Sell', zorder=5)
axes[0].set_title('Golden Cross Strategy: SPY')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Position
axes[1].fill_between(df.index, df['Position'], step='post', alpha=0.3)
axes[1].set_title('Position (1=Long, 0=Flat)')
axes[1].set_ylim(-0.1, 1.1)

# Cumulative returns
axes[2].plot(df['Market_Cumulative'], label='Buy & Hold')
axes[2].plot(df['Strategy_Cumulative'], label='Golden Cross Strategy')
axes[2].set_title('Cumulative Returns')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 1.3 Adding Transaction Costs

In [ ]:
# Transaction costs: assume 10 bps per trade (round trip)
cost_per_trade = 0.001  # 0.10%

# Identify trades (position changes)
df['Trade'] = df['Position'].diff().abs()
df['Strategy_Return_Net'] = df['Strategy_Return'] - df['Trade'] * cost_per_trade

df['Strategy_Net_Cumulative'] = (1 + df['Strategy_Return_Net']).cumprod()

n_trades = int(df['Trade'].sum())
total_cost = df['Trade'].sum() * cost_per_trade

print(f"Number of round-trip trades: {n_trades // 2}")
print(f"Total transaction costs: {total_cost:.2%}")
print(f"Strategy Return (gross): {df['Strategy_Cumulative'].iloc[-1] - 1:.2%}")
print(f"Strategy Return (net):   {df['Strategy_Net_Cumulative'].iloc[-1] - 1:.2%}")

### 1.4 Limitations of Vectorized Backtesting

- No support for realistic order execution (fills at next bar's open)
- Cannot easily model slippage, partial fills
- No portfolio/cash tracking
- Difficult to implement complex strategies (multiple assets, dynamic sizing)
- No bracket orders, trailing stops

These limitations motivate using a proper backtesting framework like **backtrader**.

## 2. Introduction to Backtrader

[Backtrader](https://www.backtrader.com/) is a feature-rich Python framework for backtesting trading strategies. It is event-driven (processes data bar by bar) and supports:

- **Multiple data feeds** (stocks, forex, futures)
- **Built-in indicators** (SMA, EMA, RSI, MACD, Bollinger Bands, etc.)
- **Realistic order execution** (market, limit, stop, bracket)
- **Commission and slippage** modeling
- **Position sizing** (sizers)
- **Performance analysis** (analyzers)
- **Optimization** (parameter grid search)
- **Plotting** (built-in charting)

### 2.1 Architecture Overview

Backtrader has four key components:

1. **Cerebro** (the brain): The engine that runs the backtest
2. **Strategy**: Your trading logic
3. **Data Feed**: Price/volume data
4. **Broker**: Simulates order execution, tracks portfolio

In [ ]:
import backtrader as bt

print(f"Backtrader version: {bt.__version__}")

### 2.2 Your First Backtrader Script

Let us walk through the minimal steps to run a backtest.

In [ ]:
# Step 1: Create a Cerebro engine
cerebro = bt.Cerebro()

# Step 2: Create a data feed from pandas DataFrame
spy = yf.download('SPY', start='2020-01-01', end='2024-12-31')
spy.columns = spy.columns.droplevel('Ticker')

bt_data = bt.feeds.PandasData(dataname=spy)

# Step 3: Add data to cerebro
cerebro.adddata(bt_data)

# Step 4: Set initial cash
cerebro.broker.setcash(100000.0)

# Step 5: Run (with no strategy, just buy & hold effectively nothing)
print(f"Starting Portfolio Value: ${cerebro.broker.getvalue():,.2f}")
cerebro.run()
print(f"Final Portfolio Value:    ${cerebro.broker.getvalue():,.2f}")

### 2.3 Writing a Strategy Class

A strategy in backtrader is a Python class that inherits from `bt.Strategy`. The key methods are:

- `__init__(self)`: Called once. Define indicators and variables here.
- `next(self)`: Called on every bar. This is where your trading logic goes.
- `notify_order(self, order)`: Called when an order changes state (submitted, completed, cancelled).
- `notify_trade(self, trade)`: Called when a trade is opened or closed.

#### Accessing Data
- `self.data.close[0]` = current close price
- `self.data.close[-1]` = previous close price
- `self.data.open[0]` = current open price
- `self.data.volume[0]` = current volume

In [ ]:
class PrintCloseStrategy(bt.Strategy):
    """A minimal strategy that just prints closing prices."""
    
    def __init__(self):
        pass
    
    def next(self):
        # Print close price for the last 5 bars only (to keep output short)
        if len(self) >= len(self.data) - 5:
            print(f"Date: {self.data.datetime.date(0)}, Close: {self.data.close[0]:.2f}")

# Run
cerebro = bt.Cerebro()
cerebro.adddata(bt.feeds.PandasData(dataname=spy))
cerebro.addstrategy(PrintCloseStrategy)
cerebro.run()

### 2.4 Placing Orders

In the `next()` method, you use:
- `self.buy()` — Place a buy (market) order
- `self.sell()` — Place a sell (market) order
- `self.close()` — Close the current position
- `self.order` — Track pending orders

By default, orders execute at the **next bar's open** price — this prevents look-ahead bias.

In [ ]:
class BuyAndHoldStrategy(bt.Strategy):
    """Buy on first bar, hold until end."""
    
    def __init__(self):
        self.order = None
    
    def notify_order(self, order):
        if order.status in [order.Completed]:
            if order.isbuy():
                print(f"BUY EXECUTED: Price={order.executed.price:.2f}, "
                      f"Size={order.executed.size}, Cost={order.executed.value:.2f}")
            elif order.issell():
                print(f"SELL EXECUTED: Price={order.executed.price:.2f}, "
                      f"Size={order.executed.size}")
        self.order = None
    
    def next(self):
        if self.order:
            return
        
        if not self.position:
            # Buy with all available cash
            size = int(self.broker.getcash() / self.data.close[0])
            self.order = self.buy(size=size)

# Run
cerebro = bt.Cerebro()
cerebro.adddata(bt.feeds.PandasData(dataname=spy))
cerebro.addstrategy(BuyAndHoldStrategy)
cerebro.broker.setcash(100000.0)

print(f"Starting Value: ${cerebro.broker.getvalue():,.2f}")
cerebro.run()
print(f"Final Value:    ${cerebro.broker.getvalue():,.2f}")

## 3. Golden Cross Strategy in Backtrader

Now let us implement the same Golden Cross (SMA 50/200 crossover) strategy using backtrader.

### 3.1 Strategy Implementation

In [ ]:
class GoldenCrossStrategy(bt.Strategy):
    """
    Golden Cross / Death Cross strategy.
    Buy when SMA(50) crosses above SMA(200).
    Sell when SMA(50) crosses below SMA(200).
    """
    params = dict(
        short_period=50,
        long_period=200,
        printlog=True,
    )
    
    def __init__(self):
        # Calculate indicators
        self.sma_short = bt.indicators.SimpleMovingAverage(
            self.data.close, period=self.params.short_period)
        self.sma_long = bt.indicators.SimpleMovingAverage(
            self.data.close, period=self.params.long_period)
        
        # Crossover signal: +1 when short crosses above long, -1 when below
        self.crossover = bt.indicators.CrossOver(self.sma_short, self.sma_long)
        
        self.order = None
        self.trade_count = 0
    
    def log(self, txt, dt=None):
        if self.params.printlog:
            dt = dt or self.data.datetime.date(0)
            print(f"{dt}: {txt}")
    
    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            return
        
        if order.status in [order.Completed]:
            if order.isbuy():
                self.log(f"BUY @ {order.executed.price:.2f}, Size={order.executed.size}")
            elif order.issell():
                self.log(f"SELL @ {order.executed.price:.2f}, Size={order.executed.size}")
        
        elif order.status in [order.Canceled, order.Margin, order.Rejected]:
            self.log("Order Canceled/Margin/Rejected")
        
        self.order = None
    
    def notify_trade(self, trade):
        if trade.isclosed:
            self.trade_count += 1
            self.log(f"TRADE #{self.trade_count} CLOSED: "
                     f"PnL={trade.pnl:.2f}, Net PnL={trade.pnlcomm:.2f}")
    
    def next(self):
        if self.order:
            return
        
        if not self.position:
            if self.crossover > 0:  # Golden Cross: BUY
                size = int(self.broker.getcash() / self.data.close[0])
                self.order = self.buy(size=size)
        else:
            if self.crossover < 0:  # Death Cross: SELL
                self.order = self.close()
    
    def stop(self):
        self.log(f"=== Strategy Summary: {self.trade_count} trades completed ===")

### 3.2 Running the Backtest

In [ ]:
# Create Cerebro
cerebro = bt.Cerebro()

# Add data (SPY 2010-2024)
spy_data = yf.download('SPY', start='2010-01-01', end='2024-12-31')
spy_data.columns = spy_data.columns.droplevel('Ticker')
bt_feed = bt.feeds.PandasData(dataname=spy_data)
cerebro.adddata(bt_feed)

# Add strategy
cerebro.addstrategy(GoldenCrossStrategy, printlog=True)

# Set broker parameters
cerebro.broker.setcash(100000.0)
cerebro.broker.setcommission(commission=0.001)  # 0.1% per trade

# Add analyzers
cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe', riskfreerate=0.0)
cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')
cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trades')
cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')

# Run
print(f"Starting Portfolio Value: ${cerebro.broker.getvalue():,.2f}")
print("=" * 60)
results = cerebro.run()
strat = results[0]
print("=" * 60)
print(f"Final Portfolio Value:    ${cerebro.broker.getvalue():,.2f}")

### 3.3 Analyzing Results

In [ ]:
# Extract analyzer results
sharpe = strat.analyzers.sharpe.get_analysis()
dd = strat.analyzers.drawdown.get_analysis()
trades = strat.analyzers.trades.get_analysis()
returns_analysis = strat.analyzers.returns.get_analysis()

print("=" * 50)
print("PERFORMANCE SUMMARY")
print("=" * 50)
print(f"Total Return:     {returns_analysis.get('rtot', 0):.2%}")
print(f"Sharpe Ratio:     {sharpe.get('sharperatio', 'N/A')}")
print(f"Max Drawdown:     {dd.max.drawdown:.2f}%")
print(f"Max DD Duration:  {dd.max.len} bars")

print(f"\n--- Trade Statistics ---")
total = trades.get('total', {})
won = trades.get('won', {})
lost = trades.get('lost', {})
print(f"Total Trades:     {total.get('total', 0)}")
print(f"Won:              {won.get('total', 0)}")
print(f"Lost:             {lost.get('total', 0)}")
if total.get('total', 0) > 0:
    win_rate = won.get('total', 0) / total.get('total', 0)
    print(f"Win Rate:         {win_rate:.1%}")

### 3.4 Plotting

In [ ]:
# Backtrader built-in plotting
# Note: In Jupyter, this may require %matplotlib inline or widget
%matplotlib inline
cerebro.plot(style='candle', volume=False, figsize=(16, 10), iplot=False)

## 4. Understanding Backtrader's Execution Model

### 4.1 Bar Processing Flow

For each bar (day), backtrader processes in this order:

1. **New bar data is loaded** (OHLCV)
2. **Pending orders are checked** against new bar's prices
3. **Orders execute** if conditions are met (e.g., market order fills at open)
4. **Notifications** are sent (`notify_order`, `notify_trade`)
5. **`next()`** is called — your strategy logic runs
6. **New orders** placed in `next()` will execute on the **next** bar

This ensures **no look-ahead bias** — orders placed based on today's close execute at tomorrow's open.

### 4.2 Order Execution Types

| Order Type | Execution |
|:---|:---|
| `bt.Order.Market` | Next bar's open price |
| `bt.Order.Close` | Current bar's close price |
| `bt.Order.Limit` | When price reaches limit price |
| `bt.Order.Stop` | When price reaches stop price, becomes market |
| `bt.Order.StopLimit` | When price reaches stop, becomes limit order |

### 4.3 Backtesting Steps Summary

The systematic approach to building a trading strategy:

1. **Data** → Load and clean price data
2. **Indicators** → Calculate technical indicators (SMA, RSI, MACD, etc.)
3. **Signals** → Define entry/exit conditions from indicators
4. **Rules** → Translate signals into orders (buy, sell, close)
5. **Orders** → Submit to broker for execution
6. **Transactions** → Orders fill, portfolio updated
7. **Evaluation** → Analyze performance (return, drawdown, Sharpe)

### 4.4 Using Data from CSV Files

In [ ]:
# If you have a CSV file with columns: Date, Open, High, Low, Close, Volume
# You can load it like this:

# bt_data = bt.feeds.GenericCSVData(
#     dataname='my_stock.csv',
#     dtformat='%Y-%m-%d',
#     datetime=0,
#     open=1, high=2, low=3, close=4, volume=5,
#     openinterest=-1,
#     fromdate=datetime.datetime(2020, 1, 1),
#     todate=datetime.datetime(2024, 12, 31),
# )

# From pandas DataFrame (most common):
# bt_data = bt.feeds.PandasData(dataname=df)

print("Data loading examples shown above (commented out for reference)")

### 4.5 Using Parameters

In [ ]:
class ParameterizedStrategy(bt.Strategy):
    """Demonstrate how to use parameters in backtrader strategies."""
    
    params = dict(
        sma_period=20,
        rsi_period=14,
        rsi_lower=30,
        rsi_upper=70,
    )
    
    def __init__(self):
        self.sma = bt.indicators.SMA(self.data.close, period=self.p.sma_period)
        self.rsi = bt.indicators.RSI(self.data.close, period=self.p.rsi_period)
    
    def next(self):
        if not self.position:
            if self.rsi < self.p.rsi_lower and self.data.close > self.sma:
                self.buy()
        else:
            if self.rsi > self.p.rsi_upper:
                self.close()

# Usage:
# cerebro.addstrategy(ParameterizedStrategy, sma_period=30, rsi_lower=25)
print("Parameterized strategy defined. Parameters:")
print(f"  sma_period: {ParameterizedStrategy.params.sma_period}")
print(f"  rsi_period: {ParameterizedStrategy.params.rsi_period}")
print(f"  rsi_lower:  {ParameterizedStrategy.params.rsi_lower}")
print(f"  rsi_upper:  {ParameterizedStrategy.params.rsi_upper}")

## 5. Exercises

1. **Vectorized Backtest**: Implement an SMA(20) crossover strategy (buy when price > SMA(20), sell when price < SMA(20)) using only pandas for AAPL (2020-2024). Plot the equity curve.

2. **First Backtrader Strategy**: Implement a strategy in backtrader that:
   - Downloads MSFT data from 2015 to 2024
   - Buys when the 10-day SMA crosses above the 30-day SMA
   - Sells when the 10-day SMA crosses below the 30-day SMA
   - Uses $100,000 initial capital and 0.1% commission
   - Report: total return, number of trades, Sharpe ratio

3. **Buy-the-Dip Strategy**: Create a backtrader strategy that buys when the stock drops more than 3% in one day, and sells after 5 trading days. Test on SPY.

4. Compare the results of your vectorized backtest (#1) with the equivalent backtrader implementation. Are the results identical? If not, why?

---
### References
- [Backtrader Documentation](https://www.backtrader.com/docu/)
- [Backtrader Quickstart](https://www.backtrader.com/docu/quickstart/quickstart/)
- Perry Kaufman, *Trading Systems and Methods*